# **Building a Feedforward Neural Network for Sentiment Analysis**


**Objective:**
In this assignment, you will implement a feedforward neural network using TensorFlow and Keras to perform sentiment analysis on the IMDB movie review dataset. You will not only code the network and train it but also explain your design choices and discuss potential improvements.

**The Dataset:**
This is a dataset of 25,000 movies reviews from IMDb, labeled by sentiment (positive/negative). Reviews have been preprocessed, and each review is encoded as a list of word indexes (integers). For convenience, words are indexed by overall frequency in the dataset, so that for instance the integer "3" encodes the 3rd most frequent word in the dataset. Therefore, if the words "movie," "The," "excellent," and "was" are the 5th, 1st, 207th, and 86th most frequent words in the dataset, a review beginning with "The movie was excellent..." is encoded as `[1, 5, 86, 207, ...]`

This allows for quick filtering operations such as: "only consider the top 10,000 most common words, but eliminate the top 20 most common words".

In [6]:
import numpy as np
import tensorflow as tf


## Data Loading and Preprocessing
Task 1:

* Load the IMDB dataset using TensorFlow’s Keras API, and restrict the vocabulary to the 10,000 most frequent words.
  * You will need the `imdb.load_data()` method from the Keras API. Explain in a couple sentences what this function does.
  * Find what the `num_words` argument to this function does.
  * Keep in mind that this function automatically splits data into Train and Test sets. What is the difference between this function and Scikit-Learn's `train_test_split()`?



In [9]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=10000)

`imdb.load_data()`: This function loads the IMDB movie review dataset which consists of 25,000 movie reviews labeled as positive or negative.
It returns pre-processed movie reviews as sequences of word indices (integers) and their sentiment labels (0 for negative, 1 for positive).

`num_words`: integer or None. Words are ranked by how often they occur (in the training set) and only the num_words most frequent words are kept. Any less frequent word will appear as oov_char value in the sequence data. If None, all words are kept. Defaults to None.

`train_test_split()` is a general-purpose function that takes an already loaded dataset and splits it into training and testing sets with a user-defined ratio. In contrast, `imdb.load_data()` handles downloading, preprocessing into integer sequences, and returns already-split training and testing sets.

In [3]:
print(x_train.shape)
print(len(x_train[0]))

(25000,)
218


In [9]:
print(x_train.shape)
print(len(x_train[0]))

(25000,)
218


In [4]:
print(x_train[0][:10])
print(x_train[1][:10])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65]
[1, 194, 1153, 194, 8255, 78, 228, 5, 6, 1463]


In [6]:
print(x_train[0][:10])
print(x_train[1][:10])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65]
[1, 194, 1153, 194, 8255, 78, 228, 5, 6, 1463]


* Implement a function named `vectorize_sequences()` that takes as input a list of sequences (each sequence contains word indices) and outputs a corresponding multi-hot encoded NumPy array. Each vector should be of length 10,000, where each entry is 1 if the word is present in the review and 0 otherwise.

* So, for example, since `x_train[0]` begins with `[1, 14, 22, 16, ...]` vectorizing `x_train[0]` produces a vector of size 10,000 with a `1` at the 1st, 14th, 22nd, 16th, ... coordinates.

Use your function to preprocess both the training and testing datasets.

In [ ]:
def vectorize_sequences(sequences, num_words=10000):
    results = np.zeros((len(sequences), num_words))
    for i, sequence in enumerate(sequences):
        for word_index in set(sequence):
            results[i, word_index] = 1.
    return results

x_train = vectorize_sequences(x_train)
x_test = vectorize_sequences(x_test)

array([0., 1., 1., 0., 1., 1., 1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 1.,
       1., 1., 1., 0., 1., 1., 0.])

In [11]:
x_train.shape

(25000, 10000)

In [11]:
x_train.shape

(25000, 10000)

In [12]:
x_train[0,:24]

array([0., 1., 1., 0., 1., 1., 1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 1.,
       1., 1., 1., 0., 1., 1., 0.])

In [12]:
x_train[0,:24]

array([0., 1., 1., 0., 1., 1., 1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 1.,
       1., 1., 1., 0., 1., 1., 0.])

## Model Construction
Task 2:

Create a Feedforward Network (Sequential model) with the following layers:

* Input Layer: Accepts a 10,000-dimensional input. Why `10,000`?

* Three Hidden Layers: Each hidden layer should use a Dense layer with 64 neurons and the ReLU activation function.

* Output Layer: A Dense layer with a single neuron and a sigmoid activation function to output a probability for binary classification.

* Compile the model with the Adam optimizer, binary crossentropy as the loss function, and track the accuracy metric.

1. Display a summary of your model architecture. The `summary()` method might come in handy here.
2. Given that this network is fully connected, how many weights does the network have?

In [13]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(10000,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

/Users/jirissman/Documents/coursework/BU/CS767/.venv/lib/python3.9/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │       640,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 648,449 (2.47 MB)

 Trainable params: 648,449 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

The input layer needs to accept 10,000 inputs because we have set the number of words in the dataset to 10,000. Each input represents a different word in a review.

The network has 648,449 weights, including bias terms.

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,304,961 (4.98 MB)

 Trainable params: 1,304,961 (4.98 MB)

 Non-trainable params: 0 (0.00 B)

## Model Training and Evaluation
Task:

* Train your model for 10 epochs using a batch size of 512, with 20% of the training data set aside for validation.
  * When you call the `fit()` method, what do the arguments `epochs,` `batch_size,` and `validation_split` do?
* After training, evaluate your model on the test dataset and print out both the loss and the accuracy.
* On page 337 of the textbook, the author describes how to save and restore a TF model. Some of the information therein is deprecated. Briefly explain how the `.save()` method works in Keras 3 compared to the textbook. When a model is saved in the `.keras` format, what information about the model gets saved?

In [16]:
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=512,
    validation_split=0.2
)

Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7141 - loss: 0.5556 - val_accuracy: 0.8814 - val_loss: 0.3065
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9309 - loss: 0.1862 - val_accuracy: 0.8842 - val_loss: 0.3001
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9630 - loss: 0.1170 - val_accuracy: 0.8788 - val_loss: 0.3664
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9807 - loss: 0.0675 - val_accuracy: 0.8760 - val_loss: 0.4335
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9897 - loss: 0.0377 - val_accuracy: 0.8718 - val_loss: 0.5358
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9961 - loss: 0.0176 - val_accuracy: 0.8738 - val_loss: 0.6198
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - accuracy: 0.9988 - loss: 0.0089 - val_accuracy: 0.8720 - val_loss: 0.6652
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - accuracy: 0.9977 - loss: 0.0114 - val_accuracy: 0.8700 - va

`fit()` **Method Parameters**

- `epochs`: Specifies how many times the learning algorithm will process the entire training dataset. Each epoch involves one complete pass through all training samples, allowing the model to iteratively improve by adjusting weights based on calculated errors.

- `batch_size`: Determines how many training samples are processed before the model's internal parameters are updated. With a batch_size of 512, the model processes 512 samples at a time before making weight adjustments. Larger batches enable faster training but require more memory, while smaller batches provide more frequent updates.

- `validation_split`: Defines what fraction of training data should be reserved for validation during training. A value of 0.2 means 20% of training data is used to evaluate model performance after each epoch without affecting weight updates. This helps monitor whether the model is learning effectively or overfitting.

In [19]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 616us/step - accuracy: 0.8535 - loss: 0.9081
Test loss: 0.9076
Test accuracy: 0.8540


In Keras 3, the `model.save()` method defaults to the `.keras` format, which is a zip archive containing the model's configuration, weights, and optimizer state.

The textbook describes saving with `save_format="tf"`, which uses the TensorFlow SavedModel directory structure, or `save_format="h5"` which uses the older HDF5 file format. While these formats are still available in Keras 3, the recommended and default format is now `.keras`.

When saving a model using the `.keras` format, the following information is included:

- The model's architecture and configuration.
- The values of the model's weights.
- The model's compilation information, including the optimizer and its state, losses, and metrics.